# Prediction Notebook

This notebook demonstrates how to use the trained model to make predictions.

## Contents
1. Load the trained model
2. Make a single prediction
3. Visualize prediction with price chart
4. Batch predictions
5. Export predictions

In [ ]:
# Add parent directory to path for imports
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.config import MODELS_DIR, DEVICE

print(f'Device: {DEVICE}')
print(f'Models directory: {MODELS_DIR}')

## 1. Load the Trained Model

In [ ]:
from src.prediction.predictor import Predictor

# List available models
print('Available models:')
for model_file in MODELS_DIR.glob('*.pt'):
    print(f'  {model_file.name}')

In [ ]:
# Initialize predictor (uses best_model.pt by default)
predictor = Predictor()
print(f'Loaded model from: {predictor.model_path}')

## 2. Make a Single Prediction

In [ ]:
# Predict for a single stock
ticker = 'AAPL'
prediction = predictor.predict(ticker)

print(f'\n{"=" * 40}')
print(f'Prediction for {ticker}')
print(f'{"=" * 40}')
print(f'Signal: {prediction.signal}')
print(f'Confidence: {prediction.confidence:.1%}')
print(f'Bullish probability: {prediction.bullish_prob:.1%}')
print(f'Bearish probability: {prediction.bearish_prob:.1%}')
print(f'Horizon: T+{prediction.horizon} days')
print(f'Timestamp: {prediction.timestamp}')

In [ ]:
# Try different stocks
test_tickers = ['MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA']

print(f'\nPredictions for popular tech stocks:')
print(f'{"Ticker":<8} {"Signal":<6} {"Confidence":>12}')
print('-' * 30)

for ticker in test_tickers:
    try:
        pred = predictor.predict(ticker)
        print(f'{pred.ticker:<8} {pred.signal:<6} {pred.confidence:>11.1%}')
    except Exception as e:
        print(f'{ticker:<8} ERROR: {e}')

## 3. Visualize Prediction with Price Chart

In [ ]:
from src.data.fetcher import fetch_stock_data
from src.visualization.plots import plot_stock_price
from datetime import datetime, timedelta

# Fetch recent price data
ticker = 'AAPL'
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')

df = fetch_stock_data(ticker, start_date, end_date)
print(f'Fetched {len(df)} days of data for {ticker}')

In [ ]:
# Get prediction
prediction = predictor.predict(ticker)

# Plot price with prediction
fig = plot_stock_price(
    prices=df['Close'].values,
    dates=df.index.tolist(),
    prediction=prediction.signal,
    confidence=prediction.confidence,
    title=f'{ticker} Price History with T+{prediction.horizon} Prediction'
)
plt.show()

In [ ]:
# Create a more detailed visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

# Price chart
ax1 = axes[0]
ax1.plot(df.index, df['Close'], 'b-', linewidth=1.5, label='Close Price')
ax1.fill_between(df.index, df['Low'], df['High'], alpha=0.2, label='High-Low Range')

# Mark the window used for prediction
window_start = len(df) - 256
ax1.axvspan(df.index[window_start], df.index[-1], alpha=0.1, color='yellow', label='Prediction Window')

# Add prediction annotation
color = 'green' if prediction.signal == 'BUY' else 'red'
arrow = '↑' if prediction.signal == 'BUY' else '↓'
ax1.annotate(
    f'{prediction.signal} {arrow}\n{prediction.confidence:.1%}',
    xy=(df.index[-1], df['Close'].iloc[-1]),
    xytext=(30, 0),
    textcoords='offset points',
    fontsize=14,
    fontweight='bold',
    color=color,
    arrowprops=dict(arrowstyle='->', color=color),
)

ax1.set_ylabel('Price ($)')
ax1.set_title(f'{ticker} - T+{prediction.horizon} Day Prediction: {prediction.signal}')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Volume chart
ax2 = axes[1]
ax2.bar(df.index, df['Volume'] / 1e6, alpha=0.7, width=1)
ax2.set_ylabel('Volume (M)')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Batch Predictions

Generate predictions for multiple stocks at once.

In [ ]:
from src.data.fetcher import fetch_sp500_tickers

# Get S&P 500 tickers
sp500_tickers = fetch_sp500_tickers()
print(f'Total S&P 500 stocks: {len(sp500_tickers)}')

In [ ]:
# For demonstration, use a subset of stocks
# Change to sp500_tickers for full analysis (will take longer)
sample_tickers = sp500_tickers[:50]  # First 50 stocks

print(f'Generating predictions for {len(sample_tickers)} stocks...')
predictions = predictor.predict_batch(sample_tickers, verbose=True)

In [ ]:
# Summary statistics
buy_count = sum(1 for p in predictions if p.signal == 'BUY')
sell_count = sum(1 for p in predictions if p.signal == 'SELL')
avg_confidence = np.mean([p.confidence for p in predictions])

print(f'\n{"=" * 40}')
print('Batch Prediction Summary')
print(f'{"=" * 40}')
print(f'Total predictions: {len(predictions)}')
print(f'BUY signals: {buy_count} ({buy_count/len(predictions):.1%})')
print(f'SELL signals: {sell_count} ({sell_count/len(predictions):.1%})')
print(f'Average confidence: {avg_confidence:.1%}')

In [ ]:
from src.prediction.predictor import rank_predictions, predictions_to_dataframe, get_top_signals

# Top BUY signals
top_buys = get_top_signals(predictions, n=10, signal_type='BUY')

print('\nTop 10 BUY Signals:')
print(predictions_to_dataframe(top_buys).to_string(index=False))

In [ ]:
# Top SELL signals
top_sells = get_top_signals(predictions, n=10, signal_type='SELL')

print('\nTop 10 SELL Signals:')
print(predictions_to_dataframe(top_sells).to_string(index=False))

In [ ]:
# Visualize confidence distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Signal distribution
ax1 = axes[0]
signals = ['BUY', 'SELL']
counts = [buy_count, sell_count]
colors = ['green', 'red']
ax1.bar(signals, counts, color=colors, alpha=0.7)
ax1.set_ylabel('Count')
ax1.set_title('Signal Distribution')
for i, (sig, cnt) in enumerate(zip(signals, counts)):
    ax1.text(i, cnt + 1, str(cnt), ha='center', fontsize=12)

# Confidence distribution
ax2 = axes[1]
buy_confs = [p.confidence for p in predictions if p.signal == 'BUY']
sell_confs = [p.confidence for p in predictions if p.signal == 'SELL']

ax2.hist(buy_confs, bins=20, alpha=0.6, label='BUY', color='green')
ax2.hist(sell_confs, bins=20, alpha=0.6, label='SELL', color='red')
ax2.set_xlabel('Confidence')
ax2.set_ylabel('Count')
ax2.set_title('Confidence Distribution by Signal')
ax2.legend()

plt.tight_layout()
plt.show()

## 5. Export Predictions

In [ ]:
from src.prediction.predictor import export_predictions
from pathlib import Path

# Create output directory
output_dir = Path('../data/predictions')
output_dir.mkdir(parents=True, exist_ok=True)

# Export all predictions to CSV
csv_path = output_dir / 'predictions.csv'
export_predictions(predictions, csv_path, format='csv')
print(f'Saved predictions to {csv_path}')

# Export to JSON as well
json_path = output_dir / 'predictions.json'
export_predictions(predictions, json_path, format='json')
print(f'Saved predictions to {json_path}')

In [ ]:
# Read back and display
df_predictions = pd.read_csv(csv_path)
print(f'\nExported {len(df_predictions)} predictions:')
df_predictions.head(10)

## Summary

This notebook demonstrated:

1. ✓ Loading a trained model with the Predictor class
2. ✓ Making predictions for single stocks
3. ✓ Visualizing predictions with price charts
4. ✓ Batch predictions across multiple stocks
5. ✓ Ranking and filtering signals by confidence
6. ✓ Exporting predictions to CSV/JSON

**Note:** Predictions are based on historical patterns and should not be used as financial advice. Always do your own research before making investment decisions.